# `Validator` - Validation des données agricoles

**Module :** `kadi.kidas.validator`  
**Classe :** `Validator`

---

## Introduction

`Validator` est la classe de validation de la qualité des données agricoles. Elle vérifie :

- **Schéma** : présence et types des colonnes obligatoires
- **Plages de valeurs** : rendements, prix, températures dans des bornes agronomiques
- **Coordonnées GPS** : dans la bounding box du Bénin (`lat: 2.5–12.5, lon: 0.8–3.8`)
- **Unicité** : pas de doublons sur des combinaisons clés (`marche + date + culture`)
- **Score de qualité global** : composites sur 3 dimensions (complétude, cohérence, précision)
- **Rapport de validation** : synthèse consultable après toutes les vérifications

## 1. Données de démonstration

In [ ]:
import os
import pandas as pd

import numpy as np
import pandas as pd

from kadi.kidas.validator import Validator

# Dataset agricole réaliste (prix marchés Bénin)
df = pd.DataFrame({
    'culture':         ['maïs', 'niébé', 'igname', 'sorgho', 'manioc'],
    'marche':          ['Dantokpa', 'Parakou', 'Bohicon', 'Kandi', 'Cotonou'],
    'prix_xof':        [350, 500, 250, 300, 180],
    'rendement_kg':    [1500.0, 800.0, 2000.0, 1200.0, 3500.0],
    'lat':             [6.366, 9.337, 7.181, 11.133, 6.366],
    'lon':             [2.437, 2.629, 2.067, 2.740, 2.420],
    'date':            pd.to_datetime(['2024-01-15', '2024-01-16', '2024-01-17', '2024-01-18', '2024-01-19']),
})

# Dataset avec erreurs intentionnelles pour la démonstration
df_avec_erreurs = pd.DataFrame({
    'culture':         ['maïs', 'niébé', None, 'sorgho'],
    'prix_xof':        [350, -50, 99999, 300],       # -50 et 99999 : invalides
    'lat':             [6.366, 9.337, 50.0, 11.133],  # 50.0 : hors bbox Bénin
    'lon':             [2.437, 2.629, 2.067, 2.740],
})

print(f"Dataset valide : {len(df)} lignes")
df

## 2. Initialisation de `Validator`

In [ ]:
validator = Validator(df)
print(repr(validator))

## 3. `check_schema()` - Validation du schéma de données

In [ ]:
# Définition du schéma attendu : nom_colonne → type
schema_attendu = {
    'culture':      'str',
    'marche':       'str',
    'prix_xof':     'float',
    'rendement_kg': 'float',
    'lat':          'float',
    'lon':          'float',
    'date':         'datetime',
}

# Validation du schéma
est_valide, erreurs = validator.check_schema(schema_attendu)
print(f"Schéma valide : {est_valide}")
if erreurs:
    for e in erreurs:
        print(f"  - {e}")
else:
    print("  Aucune erreur de schéma !")

In [ ]:
# Test avec une colonne manquante
schema_invalide = {
    'culture': 'str',
    'colonne_manquante': 'float',  # N'existe pas
    'prix_xof': 'int',             # Mauvais type (c'est un float)
}

validator2 = Validator(df)
valide, erreurs = validator2.check_schema(schema_invalide)
print(f"Schéma invalide : {valide}")
for e in erreurs:
    print(f"  - {e}")

## 4. `check_ranges()` - Validation des plages de valeurs agronomiques

In [ ]:
# Définition des plages acceptables pour les données agricoles Bénin
plages = {
    'prix_xof':     (0, 10000),      # Prix : entre 0 et 10 000 XOF/kg
    'rendement_kg': (0, 50000),      # Rendement : entre 0 et 50 tonnes/ha
}

validator = Validator(df)
est_valide, df_hors_bornes = validator.check_ranges(plages)
print(f"Toutes valeurs dans les plages : {est_valide}")
if len(df_hors_bornes) > 0:
    print(df_hors_bornes)

In [ ]:
# Test avec des données erronées
validator_err = Validator(df_avec_erreurs)
est_valide_err, df_hors = validator_err.check_ranges({'prix_xof': (0, 5000)})
print(f"Valeurs hors bornes : {est_valide_err}")
print(df_hors[['culture', 'prix_xof']])

## 5. `check_coords()` - Validation des coordonnées GPS (bbox Bénin)

In [ ]:
# Bbox Bénin par défaut : lat ∈ [2.5, 12.5], lon ∈ [0.8, 3.8]
validator = Validator(df)
est_valide, df_invalides = validator.check_coords('lat', 'lon')
print(f"Coordonnées dans la bbox Bénin : {est_valide}")
if len(df_invalides) == 0:
    print("  Tous les points GPS sont dans le territoire béninois.")

In [ ]:
# Test avec des coordonnées hors Bénin (lat=50.0 = Europe)
validator_err = Validator(df_avec_erreurs)
est_valide_err, df_hors_bbox = validator_err.check_coords('lat', 'lon')
print(f"Coordonnées hors bbox Bénin : {est_valide_err}")
print(f"Lignes concernées : {len(df_hors_bbox)}")
if len(df_hors_bbox) > 0:
    print(df_hors_bbox[['culture', 'lat', 'lon']])

In [ ]:
# La validation géographique utilise l’emprise Bénin par défaut.
validator = Validator(df)
valide, df_invalides = validator.check_coords(lat="lat", lon="lon")
print(f"Coordonnées dans le Bénin : {valide}")

## 6. `check_unique()` - Validation de l'unicité des combinaisons clés

In [ ]:
# Vérification de l'unicité de la combinaison (marche + culture)
validator = Validator(df)
est_unique, df_doublons = validator.check_unique(['marche', 'culture'])
print(f"Combinaison (marche, culture) unique : {est_unique}")
if len(df_doublons) > 0:
    print(df_doublons[['marche', 'culture']])

In [ ]:
# Démonstration avec un DataFrame contenant des doublons
df_dup = pd.concat([df, df.iloc[[0]]], ignore_index=True)
validator_dup = Validator(df_dup)
est_unique_dup, df_dup_detectes = validator_dup.check_unique(['marche', 'culture'])
print(f"Combinaison unique (avec doublons) : {est_unique_dup}")
print(f"Lignes dupliquées : {len(df_dup_detectes)}")

## 7. `quality_score()` - Score de qualité global

Le score est calculé sur **3 dimensions** :

| Dimension | Calcul |
|---|---|
| Complétude | `1 - (NaN / total)` |
| Cohérence | Ratio colonnes types corrects |
| Précision | Ratio valeurs dans les plages |

Le score global est la **moyenne pondérée** des 3 dimensions.

In [ ]:
validator = Validator(df)
score = validator.quality_score()

print("=== Score de Qualité ===")
print(f"Score global     : {score['overall']:.2%}")
print(f"Complétude       : {score['completeness']:.2%}")
print(f"Cohérence        : {score['consistency']:.2%}")
print(f"Précision        : {score['accuracy']:.2%}")
print(f"\nDétail par colonne :")
for col, detail in score['columns'].items():
    print(f"  {col:15s} : complétude={detail:.2%}")

In [ ]:
# Comparaison avec les données erronées
validator_err = Validator(df_avec_erreurs)
score_err = validator_err.quality_score()
print(f"Score global (données erronées) : {score_err['overall']:.2%}")
print(f"Complétude                      : {score_err['completeness']:.2%}")

## 8. `report()` - Rapport complet de validation

In [ ]:
# Exécution de plusieurs validations avant de générer le rapport
validator = Validator(df)
validator.check_schema({'culture': 'str', 'prix_xof': 'float'})
validator.check_ranges({'prix_xof': (0, 10000)})
validator.check_coords('lat', 'lon')
validator.quality_score()

rapport = validator.report()

print("=== Rapport de Validation ===")
print(f"Validations effectuées : {len(rapport['validations'])}")
for v in rapport['validations']:
    print(f"  - {v['type']:20s} : {'OK' if v['valide'] else 'ECHEC'}")

if 'quality_score' in rapport:
    print(f"\nScore global : {rapport['quality_score']['overall']:.2%}")

## Récapitulatif des méthodes

| Méthode | Description |
|---|---|
| `check_schema(schema)` | Vérifie présence et types des colonnes |
| `check_ranges(ranges)` | Détecte les valeurs hors bornes agronomiques |
| `check_coords(lat, lon, ...)` | Valide les GPS contre la bbox Bénin |
| `check_unique(columns)` | Détecte les doublons sur combinaisons clés |
| `quality_score()` | Score composite 0–1 (complétude, cohérence, précision) |
| `report()` | Synthèse de toutes les validations effectuées |